# 데이터 분석 — 전체 기간 (2024~2026)

2026-10-12 DB 갱신으로 2024·2025년 데이터가 들어왔고, 2026년 불량 집계 일부가 수정되었다.
4G100 CUP 계열 품명(5가지 표기)을 하나로 보고 로트번호로 묶어 전체 기간을 다시 본다. 모델링은 하지 않는다.

DB 에서 바로 읽으므로 다시 실행하면 그 시점의 데이터로 갱신된다.

In [ ]:
import sys, os, warnings
sys.path.insert(0, '/data/Project/boyoung/pipeline')
warnings.filterwarnings('ignore')

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
plt.rc('font', family='NanumGothic'); plt.rc('axes', unicode_minus=False)
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)

import db, preprocess as P

A = P.PRODUCT_ALIASES
fo = db.table('foundry_test_result'); fo['dt'] = pd.to_datetime(fo['real_date'])
bad = db.table('bad_statistics');     bad['td'] = pd.to_datetime(bad['target_date'])
for c in ['production_qty', 'bad_qty'] + list(P.DEFECTS) + ['bad_mold']:
    bad[c] = pd.to_numeric(bad[c], errors='coerce').fillna(0)
fo['yr'], bad['yr'] = fo['dt'].dt.year, bad['td'].dt.year
fo['k'], bad['k'] = P.lot_key(fo['lot_no']), P.lot_key(bad['lot_no'])

d, rep = P.build()
d = d.sort_values('target_date').reset_index(drop=True)
d['yr'] = d['target_date'].dt.year

print('DB 기준', pd.Timestamp.now().strftime('%Y-%m-%d %H:%M'))
db.freshness()

## 1. 데이터 현황

In [ ]:
print('4G100 계열 품명 표기 (행 수)')
t = pd.concat({'주물사 시험': fo[fo['product_name'].isin(A)].groupby(['yr', 'product_name']).size(),
               '불량 집계': bad[bad['product_name'].isin(A)].groupby(['yr', 'product_name']).size()}, axis=1)
print(t.fillna(0).astype(int).to_string()); print()

b4 = bad[bad['product_name'].isin(A)]; f4 = fo[fo['product_name'].isin(A)]
s = pd.DataFrame({
    '주물사 시험 로트': f4.groupby('yr')['k'].nunique(),
    '불량 실적 로트': b4.groupby('yr')['k'].nunique(),
    '연결된 로트': d.groupby('yr').size(),
    '벤토나이트 기록률': f4.groupby('yr')['bent_val'].apply(lambda x: x.notna().mean()),
})
s['연결률'] = s['연결된 로트'] / s['불량 실적 로트']
print(s.assign(**{'연결률': s['연결률'].map('{:.0%}'.format),
                  '벤토나이트 기록률': s['벤토나이트 기록률'].map('{:.0%}'.format)}).to_string())
print('\n전체', rep)

## 2. 로트 불량률 분포

In [ ]:
TH = d['bad_rate'].mean()
desc = d.groupby('yr')['bad_rate'].describe()[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']]
desc['기준 3.38% 초과'] = d.groupby('yr')['bad_rate'].apply(lambda x: (x > TH).mean())
desc['계획서 2.95% 초과'] = d.groupby('yr')['bad_rate'].apply(lambda x: (x > 2.95).mean())
print(f'전체 평균(기준) {TH:.2f}%  ·  로트 {len(d)}개'); print(desc.round(2).to_string())

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.boxplot([d.loc[d['yr'] == y, 'bad_rate'] for y in (2024, 2025, 2026)], tick_labels=['2024', '2025', '2026'], orientation='horizontal')
ax.axvline(TH, ls='--', c='gray'); ax.set_xlabel('로트 불량률 (%)'); ax.set_title('연도별 로트 불량률')
plt.tight_layout(); plt.show()

## 3. 월별 추이 — 4G100 과 다른 제품

In [ ]:
T = list(P.DEFECTS) + ['bad_mold']
bb = bad[bad['production_qty'] >= 500].copy()            # 시험 생산 소량 로트 제외
bb['is4'] = bb['product_name'].isin(A); bb['ym'] = bb['td'].dt.to_period('M')
rate = lambda g: 100 * g['bad_qty'].sum() / g['production_qty'].sum()
per = lambda c: (lambda g: 1000 * g[c].sum() / g['production_qty'].sum())
g4, go = bb[bb['is4']].groupby('ym'), bb[~bb['is4']].groupby('ym')
M = pd.DataFrame({'4G100 불량률': g4.apply(rate), '로트': g4.size(), '다른 제품 불량률': go.apply(rate),
                  '몰드크랙‰': g4.apply(per('bad_crack')), '사혼입‰': g4.apply(per('bad_mix')),
                  '중자‰': g4.apply(per('bad_core')),
                  '미분류‰': g4.apply(lambda g: 1000 * (g['bad_qty'].sum() - g[T].sum().sum()) / g['production_qty'].sum())})
f4m = f4.assign(ym=f4['dt'].dt.to_period('M'),
                bent=pd.to_numeric(f4['bent_val'].astype(str).str.extract(r'^\s*(\d+(?:\.\d+)?)\s*/')[0], errors='coerce'))
for c in ['perm_val', 'fill_val', 'moist_val', 'cstr_val', 'sand_temp']:
    f4m[c] = pd.to_numeric(f4m[c], errors='coerce').replace(0, np.nan)
M = M.join(f4m.groupby('ym').agg(벤토나이트=('bent', 'mean'), 충진율=('fill_val', 'mean'), 통기도=('perm_val', 'mean'),
                                  수분=('moist_val', 'mean'), 압축강도=('cstr_val', 'mean'), 모래온도=('sand_temp', 'mean')))
M = M.dropna(subset=['4G100 불량률'])
print(M.round(1).to_string())

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
x = M.index.to_timestamp()
ax[0].plot(x, M['4G100 불량률'], marker='o', label='4G100'); ax[0].plot(x, M['다른 제품 불량률'], marker='s', ls='--', label='다른 제품')
ax[0].set_ylabel('불량률 (%)'); ax[0].legend(); ax[0].set_title('월별 불량률')
ax[1].plot(x, M['몰드크랙‰'], marker='o', label='몰드크랙'); ax[1].plot(x, M['사혼입‰'], marker='s', label='사혼입')
ax[1].plot(x, M['중자‰'], marker='^', label='중자삽입불량'); ax[1].set_ylabel('1,000개당'); ax[1].legend(); ax[1].set_title('4G100 주요 불량 유형')
plt.tight_layout(); plt.show()

## 4. 불량 유형

In [ ]:
NAMES = P.DEFECTS | {'bad_mold': '형빼기'}
y4 = b4[b4['production_qty'] > 0].groupby('yr')[T + ['bad_qty']].sum()
y4['미분류'] = y4['bad_qty'] - y4[T].sum(axis=1)
cnt = y4.rename(columns=NAMES)[[NAMES[t] for t in T] + ['미분류', 'bad_qty']].rename(columns={'bad_qty': '불량 합계'})
print('불량 수량'); print(cnt.T.astype(int).to_string()); print()
print('불량 중 비중 (%)'); print((100 * cnt.drop(columns='불량 합계').div(cnt['불량 합계'], axis=0)).T.round(1).to_string())

## 5. 모래 시험 항목 — 연도별

In [ ]:
ITEMS = {'벤토나이트': 'bentonite', '모래온도': 'sand_temp_mean'} | {n: f'{p}_val_mean' for p, n in P.MEASURES if p != 'sand_temp'}
print('로트 평균값의 연도별 평균')
print(d.groupby('yr')[list(ITEMS.values())].mean().rename(columns={v: k for k, v in ITEMS.items()}).T.round(2).to_string()); print()

out = {}
for p, n in P.MEASURES:
    if p == 'sand_temp':
        continue
    v, lo, hi = (pd.to_numeric(f4[f'{p}_{s}'], errors='coerce') for s in ('val', 'min', 'max'))
    ok = v.notna() & (v != 0)
    out[n] = ((v < lo) | (v > hi))[ok].groupby(f4['yr'][ok]).mean()
print('규격 이탈률 (시험 행 기준)'); print(pd.DataFrame(out).T.map('{:.0%}'.format).to_string())

## 6. 불량률과 모래 시험 항목의 상관

연도마다 불량 수준이 달라, 3년을 섞은 상관과 같은 해 안의 상관이 다를 수 있다. 둘을 나란히 본다 (순위 상관).

In [ ]:
sp = lambda a, b: a.corr(b, method='spearman')
rows = []
for n, c in ITEMS.items():
    x = d[c]; r = {'항목': n, '전체': sp(x, d['bad_rate']), '생산 순서': sp(x, pd.Series(range(len(d))))}
    for y in (2024, 2025, 2026):
        g = d[d['yr'] == y]
        r[str(y)] = sp(g[c], g['bad_rate']) if g[c].notna().sum() >= 20 else np.nan
    r['같은 해 안'] = x.groupby(d['yr']).rank(pct=True).corr(d['bad_rate'].groupby(d['yr']).rank(pct=True))
    m = d[['yr', 'target_date']].assign(m=d['target_date'].dt.month)
    r['같은 달 안'] = x.groupby([m['yr'], m['m']]).rank(pct=True).corr(d['bad_rate'].groupby([m['yr'], m['m']]).rank(pct=True))
    rows.append(r)
corr = pd.DataFrame(rows).set_index('항목').round(2)
print(corr.sort_values('전체', key=abs, ascending=False).to_string())

## 7. 월 단위 상관 — 불량률·몰드크랙과 함께 움직인 값

In [ ]:
rows = []
for c in ['다른 제품 불량률', '충진율', '통기도', '벤토나이트', '압축강도', '수분', '모래온도']:
    r1 = stats.spearmanr(M[c], M['4G100 불량률'], nan_policy='omit'); r2 = stats.spearmanr(M[c], M['몰드크랙‰'], nan_policy='omit')
    Md = M.groupby(M.index.year).transform(lambda s: s - s.mean())
    rows.append({'값': c, '불량률 r': r1[0], '불량률 p': r1[1], '몰드크랙 r': r2[0], '몰드크랙 p': r2[1],
                 '같은 해 안 · 불량률': stats.spearmanr(Md[c], Md['4G100 불량률'], nan_policy='omit')[0],
                 '같은 해 안 · 몰드크랙': stats.spearmanr(Md[c], Md['몰드크랙‰'], nan_policy='omit')[0]})
print(f'{len(M)}개월'); print(pd.DataFrame(rows).set_index('값').round(3).to_string())

## 8. 모래 외 변수 후보

In [ ]:
oth = bb[~bb['is4']]
def win(df, t, lo, hi, dcol='td'):
    m = df[(df[dcol] > t - pd.Timedelta(days=hi)) & (df[dcol] <= t - pd.Timedelta(days=lo))]
    return 100 * m['bad_qty'].sum() / m['production_qty'].sum() if m['production_qty'].sum() > 0 else np.nan
own = d.rename(columns={'target_date': 'td'})
V = {'4G100 최근 30일 불량률': d['target_date'].map(lambda t: win(own, t, 1, 30)),
     '다른 제품 최근 30일 불량률': d['target_date'].map(lambda t: win(oth, t, 1, 30)),
     '다른 제품 같은 주 불량률': d['target_date'].map(lambda t: win(oth, t, -3, 4)),
     '생산 수량': d['production_qty'], '요일': d['target_date'].dt.dayofweek,
     '직전 로트와 간격(일)': d['target_date'].diff().dt.days}
rows = []
for n, x in V.items():
    x = pd.Series(x, index=d.index).astype(float); r = {'변수': n, '전체': sp(x, d['bad_rate'])}
    for y in (2024, 2025, 2026):
        g = d['yr'] == y; r[str(y)] = sp(x[g], d['bad_rate'][g])
    r['같은 해 안'] = x.groupby(d['yr']).rank(pct=True).corr(d['bad_rate'].groupby(d['yr']).rank(pct=True))
    rows.append(r)
print(pd.DataFrame(rows).set_index('변수').round(2).to_string())

## 9. 데이터 품질

In [ ]:
# ① 9/14 등록값과 현재 DB 비교 — 보고서 작성 당시 스냅샷
from pathlib import Path
SNAP = Path('../document/시간순검증-자료/데이터/bad_statistics_0914.csv')   # 저장소에는 없는 스냅샷
if SNAP.exists():
    snap = pd.read_csv(SNAP, dtype={'lot_no': str})
    cur = bad[(bad['product_name'] == P.PRODUCT) & (bad['td'] >= '2026-01-01') & (bad['td'] <= '2026-09-14')]
    agg = lambda x: x.groupby('lot_no')[['production_qty', 'bad_qty']].sum()
    mrg = agg(snap).join(agg(cur), how='outer', lsuffix='_9/14', rsuffix='_현재').reset_index()
    chg = mrg[(mrg['bad_qty_9/14'] != mrg['bad_qty_현재']) | (mrg['production_qty_9/14'] != mrg['production_qty_현재'])]
    print('① 9/14 등록 후 수량이 바뀐 로트'); print(chg.to_string(index=False)); print()
else:
    print('① 9/14 스냅샷이 없어 건너뜀'); print()

# ② 연도별 미분류 비중
print('② 미분류 비중'); print((y4['미분류'] / y4['bad_qty']).map('{:.1%}'.format).to_string()); print()

# ③ 불량 실적은 있으나 모래 시험과 연결되지 않은 로트
bk = b4.groupby('k').agg(yr=('yr', 'first'), prod=('production_qty', 'sum'))
miss = bk[~bk.index.isin(d['lot_key'])].copy()
f_oth = set(fo.loc[~fo['product_name'].isin(A), 'k'])
miss['이유'] = np.select([miss['prod'] < 500, miss.index.isin(f_oth)],
                        ['시험 생산 소량 로트', '다른 제품명으로만 시험 기록'], '모래 시험 기록 없음')
print('③ 연결되지 않은 불량 실적 로트'); print(miss.groupby(['yr', '이유']).size().unstack(fill_value=0).to_string())